In [1]:
import pandas as pd
import re
import string
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from nltk.tokenize import word_tokenize
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer

nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)
nltk.download('stopwords', quiet=True)
nltk.download('wordnet', quiet=True)
nltk.download('averaged_perceptron_tagger', quiet=True)

print("Libraries loaded successfully.")

Libraries loaded successfully.


## Load Labelled Excel Data

In [2]:
EXCEL_PATH = "labelled sentences(1).xlsx"

df = pd.read_excel(EXCEL_PATH)

print("Shape:", df.shape)
print("Columns:", df.columns.tolist())
print("\nLabel distribution:")
print(df['Label'].value_counts())
print("\nSample rows:")
df.head()

Shape: (689, 2)
Columns: ['Sentences', 'Label']

Label distribution:
Label
1    514
0    175
Name: count, dtype: int64

Sample rows:


,Sentences,Label
0,UN Humanitarian Chief Mark Lowcock today relea...,1
1,The announcement came as the World Health Orga...,0
2,The WHO has said there is still a chance of co...,1
3,"There are now cases linked to Iran in Bahrain,...",1
4,Extensive testing in clinical trials has confi...,1


## 1. Basic Preprocessing — Version 1 CSV

Steps applied:
- Lowercase all text
- Remove punctuation
- Remove numbers
- Strip extra whitespace

In [3]:
def basic_clean(text):
    text = text.lower()
    text = re.sub(r'\d+', '', text)
    text = text.translate(str.maketrans('', '', string.punctuation))
    text = re.sub(r'\s+', ' ', text).strip()
    return text

df_v1 = df.copy()
df_v1['Sentences'] = df_v1['Sentences'].apply(basic_clean)

df_v1.to_csv("sentences_v1_basic.csv", index=False)

print("Saved: sentences_v1_basic.csv")
print(f"Rows: {len(df_v1)}")
print("\nBefore vs After (first 2 rows):")
for i in range(2):
    print(f"\n  [RAW]   {df['Sentences'].iloc[i][:120]}")
    print(f"  [V1]    {df_v1['Sentences'].iloc[i][:120]}")

Saved: sentences_v1_basic.csv
Rows: 689

Before vs After (first 2 rows):

  [RAW]   UN Humanitarian Chief Mark Lowcock today released US$15 million from the Central Emergency Response Fund (CERF) to help 
  [V1]    un humanitarian chief mark lowcock today released us million from the central emergency response fund cerf to help fund 

  [RAW]   The announcement came as the World Health Organization (WHO) upgraded the global risk of the coronavirus outbreak to "ve
  [V1]    the announcement came as the world health organization who upgraded the global risk of the coronavirus outbreak to very 


## 1. Advanced Preprocessing — Version 2 CSV

Additional steps on top of V1:
- Tokenization
- Stopword removal
- Lemmatization

In [4]:
stop_words = set(stopwords.words('english'))
lemmatizer = WordNetLemmatizer()

def advanced_clean(text):
    # Start from the basic-cleaned version
    text = basic_clean(text)
    tokens = word_tokenize(text)
    tokens = [t for t in tokens if t not in stop_words]
    tokens = [lemmatizer.lemmatize(t) for t in tokens]
    return ' '.join(tokens)

df_v2 = df.copy()
df_v2['Sentences'] = df_v2['Sentences'].apply(advanced_clean)

df_v2.to_csv("sentences_v2_advanced.csv", index=False)

print("Saved: sentences_v2_advanced.csv")
print(f"Rows: {len(df_v2)}")
print("\nV1  vs  V2 (first 2 rows):")
for i in range(2):
    print(f"\n  [V1]  {df_v1['Sentences'].iloc[i][:120]}")
    print(f"  [V2]  {df_v2['Sentences'].iloc[i][:120]}")

Saved: sentences_v2_advanced.csv
Rows: 689

V1  vs  V2 (first 2 rows):

  [V1]  un humanitarian chief mark lowcock today released us million from the central emergency response fund cerf to help fund 
  [V2]  un humanitarian chief mark lowcock today released u million central emergency response fund cerf help fund global effort

  [V1]  the announcement came as the world health organization who upgraded the global risk of the coronavirus outbreak to very 
  [V2]  announcement came world health organization upgraded global risk coronavirus outbreak high ƒ top level risk assessment


## 2. Count Vectorization — Processed Text (V2)

In [5]:
processed = pd.read_csv("sentences_v2_advanced.csv")
corpus_processed = processed['Sentences'].tolist()

cv_processed = CountVectorizer()
X_cv_processed = cv_processed.fit_transform(corpus_processed)

print("=== Count Vectorizer — Processed (V2) ===")
print(f"Matrix shape : {X_cv_processed.shape}  (docs x unique tokens)")
print(f"Vocabulary size : {len(cv_processed.vocabulary_)}")
print(f"\nSample feature names (first 30):")
print(cv_processed.get_feature_names_out()[:30])

# Dense preview (first 5 docs, first 10 features)
cv_df_proc = pd.DataFrame(
    X_cv_processed[:5, :10].toarray(),
    columns=cv_processed.get_feature_names_out()[:10]
)
print("\nDocument-Term matrix preview (5 docs × 10 terms):")
cv_df_proc

=== Count Vectorizer — Processed (V2) ===
Matrix shape : (689, 3114)  (docs x unique tokens)
Vocabulary size : 3114

Sample feature names (first 30):
['aayog' 'abdulaziz' 'abhiyaan' 'ability' 'able' 'absence' 'abuse'
 'academic' 'academy' 'accelerate' 'accelerator' 'accept' 'accepted'
 'access' 'accessed' 'accessible' 'accessing' 'accessƒ' 'according'
 'account' 'accounted' 'accounting' 'accrue' 'achieve' 'achieved'
 'achievement' 'achieving' 'acog' 'acquiring' 'across']

Document-Term matrix preview (5 docs × 10 terms):


,aayog,abdulaziz,abhiyaan,ability,able,absence,abuse,academic,academy,accelerate
0,0,0,0,0,0,0,0,0,0,0
1,0,0,0,0,0,0,0,0,0,0
2,0,0,0,0,0,0,0,0,0,0
3,0,0,0,0,0,0,0,0,0,0
4,0,0,0,0,0,0,0,0,0,0


## 2. TF-IDF — Processed Text (V2)


In [6]:
tfidf_processed = TfidfVectorizer()
X_tfidf_processed = tfidf_processed.fit_transform(corpus_processed)

print("=== TF-IDF — Processed (V2) ===")
print(f"Matrix shape : {X_tfidf_processed.shape}  (docs x unique tokens)")
print(f"Vocabulary size : {len(tfidf_processed.vocabulary_)}")

# Top weighted terms across the whole corpus
import numpy as np
mean_tfidf = np.asarray(X_tfidf_processed.mean(axis=0)).flatten()
top_idx = mean_tfidf.argsort()[::-1][:20]
features = tfidf_processed.get_feature_names_out()
print("\nTop 20 terms by mean TF-IDF score (processed):")
for i, idx in enumerate(top_idx, 1):
    print(f"  {i:2}. {features[idx]:<20} {mean_tfidf[idx]:.4f}")

# Dense preview
tfidf_df_proc = pd.DataFrame(
    X_tfidf_processed[:5, :10].toarray(),
    columns=tfidf_processed.get_feature_names_out()[:10]
)
print("\nTF-IDF matrix preview (5 docs × 10 terms):")
tfidf_df_proc

=== TF-IDF — Processed (V2) ===
Matrix shape : (689, 3114)  (docs x unique tokens)
Vocabulary size : 3114

Top 20 terms by mean TF-IDF score (processed):
   1. covid                0.0364
   2. health               0.0305
   3. case                 0.0238
   4. country              0.0224
   5. new                  0.0205
   6. vaccine              0.0203
   7. world                0.0182
   8. pandemic             0.0172
   9. virus                0.0142
  10. death                0.0139
  11. disease              0.0134
  12. global               0.0132
  13. people               0.0131
  14. coronavirus          0.0129
  15. minister             0.0129
  16. week                 0.0123
  17. reported             0.0118
  18. variant              0.0116
  19. patient              0.0104
  20. public               0.0101

TF-IDF matrix preview (5 docs × 10 terms):


,aayog,abdulaziz,abhiyaan,ability,able,absence,abuse,academic,academy,accelerate
0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


## 3. Count Vectorization — Unprocessed (Raw) Text

In [7]:
corpus_raw = df['Sentences'].tolist()

cv_raw = CountVectorizer()
X_cv_raw = cv_raw.fit_transform(corpus_raw)

print("=== Count Vectorizer — Raw (Unprocessed) ===")
print(f"Matrix shape : {X_cv_raw.shape}  (docs x unique tokens)")
print(f"Vocabulary size : {len(cv_raw.vocabulary_)}")
print(f"\nProcessed vocab size : {len(cv_processed.vocabulary_)}")
print(f"Reduction after processing : {len(cv_raw.vocabulary_) - len(cv_processed.vocabulary_)} tokens removed "
      f"({(1 - len(cv_processed.vocabulary_)/len(cv_raw.vocabulary_))*100:.1f}% smaller)")

print(f"\nSample feature names (first 30):")
print(cv_raw.get_feature_names_out()[:30])

cv_df_raw = pd.DataFrame(
    X_cv_raw[:5, :10].toarray(),
    columns=cv_raw.get_feature_names_out()[:10]
)
print("\nDocument-Term matrix preview (5 docs × 10 terms):")
cv_df_raw

=== Count Vectorizer — Raw (Unprocessed) ===
Matrix shape : (689, 3526)  (docs x unique tokens)
Vocabulary size : 3526

Processed vocab size : 3114
Reduction after processing : 412 tokens removed (11.7% smaller)

Sample feature names (first 30):
['000' '0001' '007' '097' '10' '100' '102' '105' '106' '108' '109th' '11'
 '111' '118' '12' '1273' '13' '131' '132' '14' '140' '147th' '15' '155'
 '15th' '16' '165' '17' '172' '18']

Document-Term matrix preview (5 docs × 10 terms):


,000,0001,007,097,10,100,102,105,106,108
0,0,0,0,0,0,0,0,0,0,0
1,0,0,0,0,0,0,0,0,0,0
2,0,0,0,0,0,0,0,0,0,0
3,0,0,0,0,0,0,0,0,0,0
4,0,0,0,0,0,0,0,0,0,0


## 3. TF-IDF — Unprocessed (Raw) Text

In [8]:
tfidf_raw = TfidfVectorizer()
X_tfidf_raw = tfidf_raw.fit_transform(corpus_raw)

print("=== TF-IDF — Raw (Unprocessed) ===")
print(f"Matrix shape : {X_tfidf_raw.shape}  (docs x unique tokens)")
print(f"Vocabulary size : {len(tfidf_raw.vocabulary_)}")

mean_tfidf_raw = np.asarray(X_tfidf_raw.mean(axis=0)).flatten()
top_idx_raw = mean_tfidf_raw.argsort()[::-1][:20]
features_raw = tfidf_raw.get_feature_names_out()
print("\nTop 20 terms by mean TF-IDF score (raw):")
for i, idx in enumerate(top_idx_raw, 1):
    print(f"  {i:2}. {features_raw[idx]:<25} {mean_tfidf_raw[idx]:.4f}")

tfidf_df_raw = pd.DataFrame(
    X_tfidf_raw[:5, :10].toarray(),
    columns=tfidf_raw.get_feature_names_out()[:10]
)
print("\nTF-IDF matrix preview (5 docs × 10 terms):")
tfidf_df_raw

=== TF-IDF — Raw (Unprocessed) ===
Matrix shape : (689, 3526)  (docs x unique tokens)
Vocabulary size : 3526

Top 20 terms by mean TF-IDF score (raw):
   1. the                       0.0858
   2. of                        0.0603
   3. and                       0.0517
   4. to                        0.0504
   5. in                        0.0450
   6. covid                     0.0326
   7. 19                        0.0313
   8. for                       0.0297
   9. health                    0.0277
  10. that                      0.0227
  11. is                        0.0225
  12. with                      0.0222
  13. who                       0.0209
  14. on                        0.0208
  15. are                       0.0190
  16. as                        0.0185
  17. by                        0.0183
  18. cases                     0.0181
  19. new                       0.0177
  20. has                       0.0174

TF-IDF matrix preview (5 docs × 10 terms):


,000,0001,007,097,10,100,102,105,106,108
0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
